In [1]:
import os
import sys
import glob
import time
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio

warnings.filterwarnings("ignore")


# ============================================================
# 0. User-configured Kaggle paths
#    這裡保留你原本從 Kaggle input 複製的路徑，不擅自改寫。
# ============================================================

WHEEL_DIR = "/kaggle/input/datasets/kevinlin0411/kaggle-onnx-wheels/onnx_wheels"

TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
WEIGHTS_DIR = "/kaggle/input/datasets/kevinlin0411/birdclef-2026-my-models"
TAXA_CSV = "/kaggle/input/competitions/birdclef-2026/taxonomy.csv"
SAMPLE_SUBMISSION_CSV = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"

SUBMISSION_PATH = "/kaggle/working/submission.csv"

ONNX_FILE = "resnet18_v3_int8.onnx"


# ============================================================
# 1. Audio / inference parameters
# ============================================================

SAMPLE_RATE = 32000
CHUNK_DURATION = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_DURATION

FILE_DURATION = 60
FILE_SAMPLES = SAMPLE_RATE * FILE_DURATION
N_WINDOWS = FILE_DURATION // CHUNK_DURATION  # 12

BATCH_SIZE = 16
NUM_THREADS = min(4, os.cpu_count() or 1)

DEVICE = torch.device("cpu")

def configure_torch_threads():
    """
    Safe thread configuration for Kaggle notebook / submit environment.

    torch.set_num_interop_threads() can only be called before inter-op parallel work starts.
    In notebooks, previous cell execution may already initialize PyTorch parallel backend.
    Therefore, never call it naked.
    """
    try:
        torch.set_num_threads(NUM_THREADS)
        print(f"[INFO] torch.set_num_threads({NUM_THREADS}) succeeded.")
    except RuntimeError as e:
        print(f"[WARN] torch.set_num_threads failed: {repr(e)}")
        print("[WARN] Continuing with existing torch thread setting.")

    try:
        torch.set_num_interop_threads(1)
        print("[INFO] torch.set_num_interop_threads(1) succeeded.")
    except RuntimeError as e:
        print(f"[WARN] torch.set_num_interop_threads failed: {repr(e)}")
        print("[WARN] This is usually harmless in notebooks after PyTorch has already started parallel work.")
        print("[WARN] Continuing with existing interop thread setting.")

# 你的 ONNX 模型目前註解是：
# session.run(None, ...) 回傳 ['logits_class', 'logits_species', 'attention_weights']
# 因此 species logits index = 1。
SPECIES_OUTPUT_INDEX = 1


# ============================================================
# 2. Offline onnxruntime installation
# ============================================================

def ensure_onnxruntime():
    """
    Kaggle no-internet environment:
    先嘗試直接 import onnxruntime。
    若失敗，再從你掛載的 wheel dataset 離線安裝。
    """
    try:
        import onnxruntime as ort
        print(f"[OK] onnxruntime already available: version={ort.__version__}")
        return ort

    except Exception as import_error:
        print("[INFO] onnxruntime not importable. Trying offline wheel installation.")
        print(f"[INFO] Initial import error: {repr(import_error)}")
        print(f"[INFO] WHEEL_DIR = {WHEEL_DIR}")
        print(f"[INFO] WHEEL_DIR exists = {os.path.exists(WHEEL_DIR)}")

        if os.path.exists(WHEEL_DIR):
            wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "*.whl")))
            print(f"[INFO] Found {len(wheels)} wheel files.")
            for w in wheels[:10]:
                print(f"       - {os.path.basename(w)}")
        else:
            raise FileNotFoundError(f"WHEEL_DIR does not exist: {WHEEL_DIR}")

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-index",
                "--find-links",
                WHEEL_DIR,
                "onnxruntime",
            ],
            check=True,
        )

        import onnxruntime as ort
        print(f"[OK] onnxruntime installed offline: version={ort.__version__}")
        return ort


# ============================================================
# 3. Mel frontend
# ============================================================

class AudioToMelSpectrogram(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=2048,
            hop_length=512,
            n_mels=128,
            f_min=20,
            f_max=16000,
        ).to(DEVICE)

        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(
            top_db=80
        ).to(DEVICE)

    def forward(self, waveform):
        """
        Expected input:
            waveform shape = (batch, 1, CHUNK_SAMPLES)

        Output:
            mel shape usually = (batch, 1, 128, time_bins)
        """
        mel = self.mel_spec(waveform)
        mel = self.amplitude_to_db(mel)

        # 保留你原本的 global normalization 邏輯。
        # 注意：如果訓練時是 per-sample normalization，這裡才需要再改。
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return mel


# ============================================================
# 4. Diagnostics
# ============================================================

def report_path(name, path, expect_type=None):
    """
    expect_type:
        "file", "dir", or None
    """
    exists = os.path.exists(path)
    is_file = os.path.isfile(path)
    is_dir = os.path.isdir(path)

    print(f"[PATH] {name}")
    print(f"       path    = {path}")
    print(f"       exists  = {exists}")
    print(f"       is_file = {is_file}")
    print(f"       is_dir  = {is_dir}")

    if expect_type == "file" and not is_file:
        print(f"       [WARN] Expected a file but not found as file.")
    elif expect_type == "dir" and not is_dir:
        print(f"       [WARN] Expected a directory but not found as directory.")


def find_test_files(test_audio_dir):
    """
    用 set 去重，並排序，避免 .ogg / .OGG glob 重複或順序不穩。
    """
    files = glob.glob(os.path.join(test_audio_dir, "**", "*.ogg"), recursive=True)
    files += glob.glob(os.path.join(test_audio_dir, "**", "*.OGG"), recursive=True)
    files = sorted(set(files))
    return files


def validate_environment():
    """
    只做診斷，不擅自替換你的路徑。
    目的：讓你從 Kaggle log 直接看出路徑是否抓得到資料。
    """
    print("\n" + "=" * 80)
    print("Environment / path validation")
    print("=" * 80)

    report_path("WHEEL_DIR", WHEEL_DIR, expect_type="dir")
    report_path("TEST_AUDIO_DIR", TEST_AUDIO_DIR, expect_type="dir")
    report_path("WEIGHTS_DIR", WEIGHTS_DIR, expect_type="dir")
    report_path("TAXA_CSV", TAXA_CSV, expect_type="file")
    report_path("SAMPLE_SUBMISSION_CSV", SAMPLE_SUBMISSION_CSV, expect_type="file")

    onnx_path = os.path.join(WEIGHTS_DIR, ONNX_FILE)
    report_path("ONNX_MODEL", onnx_path, expect_type="file")

    test_files = find_test_files(TEST_AUDIO_DIR)
    print(f"\n[DATA] Number of test .ogg files found = {len(test_files)}")
    for f in test_files[:10]:
        print(f"       - {f}")

    if os.path.isfile(SAMPLE_SUBMISSION_CSV):
        sample_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
        print(f"\n[SAMPLE] sample_submission shape = {sample_df.shape}")
        print(f"[SAMPLE] number of columns = {len(sample_df.columns)}")
        print(f"[SAMPLE] first 8 columns = {sample_df.columns[:8].tolist()}")
        print(f"[SAMPLE] first 5 row_id = {sample_df['row_id'].head(5).tolist() if 'row_id' in sample_df.columns else 'NO row_id COLUMN'}")

    if os.path.isfile(TAXA_CSV):
        taxa_df = pd.read_csv(TAXA_CSV)
        print(f"\n[TAXA] taxonomy shape = {taxa_df.shape}")
        print(f"[TAXA] columns = {taxa_df.columns.tolist()}")
        if "primary_label" in taxa_df.columns:
            print(f"[TAXA] number of unique primary_label = {taxa_df['primary_label'].nunique()}")

    print("=" * 80 + "\n")

    return test_files


# ============================================================
# 5. Audio loading and fixed 60-second slicing
# ============================================================

def load_audio_as_fixed_60s(file_path):
    """
    Kaggle hidden test 每個檔案官方說是 60 秒。
    但 OGG decode 後 sample 數可能有微小偏差。
    所以這裡一律：
        - resample 到 32 kHz
        - stereo 轉 mono
        - 不足 60 秒補 0
        - 超過 60 秒截斷
        - 最終必定是 shape = (1, 60 * 32000)
    """
    waveform, sr = torchaudio.load(file_path)

    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 若某些 decoder 回傳空音訊，直接建立全 0 waveform，避免 row 數缺失。
    if waveform.numel() == 0 or waveform.shape[1] == 0:
        waveform = torch.zeros((1, FILE_SAMPLES), dtype=torch.float32)

    original_samples = waveform.shape[1]

    if original_samples < FILE_SAMPLES:
        pad_len = FILE_SAMPLES - original_samples
        waveform = torch.nn.functional.pad(waveform, (0, pad_len))
    elif original_samples > FILE_SAMPLES:
        waveform = waveform[:, :FILE_SAMPLES]

    assert waveform.shape == (1, FILE_SAMPLES), (
        f"Unexpected waveform shape after fixing: {waveform.shape}"
    )

    return waveform.contiguous(), sr, original_samples


def waveform_to_12_chunks(waveform):
    """
    Input:
        waveform shape = (1, FILE_SAMPLES)

    Output:
        chunks shape = (12, 1, CHUNK_SAMPLES)
    """
    chunks = waveform.view(1, N_WINDOWS, CHUNK_SAMPLES)
    chunks = chunks.squeeze(0).unsqueeze(1).contiguous()

    assert chunks.shape == (N_WINDOWS, 1, CHUNK_SAMPLES), (
        f"Unexpected chunk shape: {chunks.shape}"
    )

    return chunks


# ============================================================
# 6. ONNX inference
# ============================================================

def build_onnx_session(ort):
    onnx_path = os.path.join(WEIGHTS_DIR, ONNX_FILE)

    if not os.path.isfile(onnx_path):
        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")

    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = NUM_THREADS
    sess_options.inter_op_num_threads = 1
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(
        onnx_path,
        sess_options,
        providers=["CPUExecutionProvider"],
    )

    input_names = [x.name for x in session.get_inputs()]
    output_names = [x.name for x in session.get_outputs()]

    print("[ONNX] Session initialized.")
    print(f"[ONNX] input names  = {input_names}")
    print(f"[ONNX] output names = {output_names}")
    print(f"[ONNX] using input name = {input_names[0]}")
    print(f"[ONNX] using species output index = {SPECIES_OUTPUT_INDEX}")

    return session, input_names[0], output_names

def run_preflight_inference(session, input_name, mel_converter, num_model_classes):
    """
    Even when test_soundscapes is empty in notebook edit mode,
    run a dummy 5-second silent audio through:
        waveform -> mel -> ONNX -> sigmoid
    to verify that the model can actually execute on Kaggle CPU.
    """
    print("\n" + "=" * 80)
    print("ONNX preflight inference")
    print("=" * 80)

    dummy_waveform = torch.zeros(
        (1, 1, CHUNK_SAMPLES),
        dtype=torch.float32,
        device=DEVICE,
    )

    with torch.no_grad():
        dummy_mel = mel_converter(dummy_waveform)

    dummy_mel_np = dummy_mel.detach().cpu().numpy().astype(np.float32, copy=False)

    print(f"[PREFLIGHT] dummy waveform shape = {tuple(dummy_waveform.shape)}")
    print(f"[PREFLIGHT] dummy mel shape = {dummy_mel_np.shape}")
    print(f"[PREFLIGHT] ONNX input name = {input_name}")

    ort_outs = session.run(None, {input_name: dummy_mel_np})

    print(f"[PREFLIGHT] number of ONNX outputs = {len(ort_outs)}")
    for i, out in enumerate(ort_outs):
        print(f"[PREFLIGHT] output[{i}] shape = {np.asarray(out).shape}")

    if SPECIES_OUTPUT_INDEX >= len(ort_outs):
        raise IndexError(
            f"SPECIES_OUTPUT_INDEX={SPECIES_OUTPUT_INDEX}, "
            f"but model only returned {len(ort_outs)} outputs."
        )

    logits_species = np.asarray(ort_outs[SPECIES_OUTPUT_INDEX])

    if logits_species.shape[-1] != num_model_classes:
        raise ValueError(
            f"Species output dimension mismatch: got {logits_species.shape}, "
            f"expected last dimension = {num_model_classes}"
        )

    probs = torch.sigmoid(torch.from_numpy(logits_species)).numpy()

    if not np.isfinite(probs).all():
        raise ValueError("Preflight probabilities contain NaN or inf.")

    print(f"[PREFLIGHT] species logits shape = {logits_species.shape}")
    print(f"[PREFLIGHT] probs min = {float(probs.min())}")
    print(f"[PREFLIGHT] probs max = {float(probs.max())}")
    print("[PREFLIGHT] passed.")
    print("=" * 80 + "\n")

def predict_one_file(
    file_path,
    session,
    input_name,
    mel_converter,
    num_model_classes,
):
    """
    回傳：
        probs shape = (12, num_model_classes)
    """
    stem = os.path.splitext(os.path.basename(file_path))[0]

    try:
        waveform, original_sr, original_samples = load_audio_as_fixed_60s(file_path)
        chunks = waveform_to_12_chunks(waveform)

        probs_list = []

        with torch.no_grad():
            for start in range(0, N_WINDOWS, BATCH_SIZE):
                batch_chunks = chunks[start:start + BATCH_SIZE]

                mel_specs = mel_converter(batch_chunks)
                mel_specs_np = (
                    mel_specs
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32, copy=False)
                )

                ort_outs = session.run(None, {input_name: mel_specs_np})

                if SPECIES_OUTPUT_INDEX >= len(ort_outs):
                    raise IndexError(
                        f"SPECIES_OUTPUT_INDEX={SPECIES_OUTPUT_INDEX} but ONNX returned only {len(ort_outs)} outputs."
                    )

                logits_species_np = ort_outs[SPECIES_OUTPUT_INDEX]

                batch_probs = torch.sigmoid(
                    torch.from_numpy(logits_species_np)
                ).numpy()

                probs_list.append(batch_probs)

        probs = np.concatenate(probs_list, axis=0).astype(np.float32, copy=False)

        if probs.shape != (N_WINDOWS, num_model_classes):
            raise ValueError(
                f"{stem}: wrong prob shape {probs.shape}; expected {(N_WINDOWS, num_model_classes)}"
            )

        if not np.isfinite(probs).all():
            raise ValueError(f"{stem}: non-finite probability detected.")

        probs = np.clip(probs, 0.0, 1.0)

        return probs

    except Exception as e:
        # 不能 continue，否則 submission rows 會少。
        # 這裡用全 0 補齊該檔案的 12 rows，至少格式一定合法。
        print(f"[WARN] Failed to process {stem}: {repr(e)}")
        print(f"[WARN] Filling zeros for {stem}.")
        return np.zeros((N_WINDOWS, num_model_classes), dtype=np.float32)


# ============================================================
# 7. Submission construction and validation
# ============================================================

def load_submission_schema():
    """
    sample_submission 是 submission 欄位順序的唯一標準。
    不要在 sample_submission 讀取失敗時 fallback 成 taxonomy 字母排序。
    """
    if not os.path.isfile(SAMPLE_SUBMISSION_CSV):
        raise FileNotFoundError(
            f"sample_submission.csv not found: {SAMPLE_SUBMISSION_CSV}"
        )

    sample_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)

    if "row_id" not in sample_df.columns:
        raise ValueError("sample_submission.csv does not contain row_id column.")

    official_cols = sample_df.columns.tolist()
    target_cols = official_cols[1:]

    print(f"[SCHEMA] sample_submission shape = {sample_df.shape}")
    print(f"[SCHEMA] number of target columns = {len(target_cols)}")

    return sample_df, official_cols, target_cols


def load_model_class_names():
    """
    這裡假設你的模型輸出順序與訓練時使用的 sorted taxonomy primary_label 一致。
    如果你訓練時曾另存 class_order.json，最理想是改成讀那個 json。
    """
    if not os.path.isfile(TAXA_CSV):
        raise FileNotFoundError(f"taxonomy.csv not found: {TAXA_CSV}")

    taxa_df = pd.read_csv(TAXA_CSV)

    if "primary_label" not in taxa_df.columns:
        raise ValueError("taxonomy.csv does not contain primary_label column.")

    model_class_names = sorted(taxa_df["primary_label"].astype(str).unique().tolist())

    print(f"[CLASSES] number of model classes from taxonomy = {len(model_class_names)}")
    print(f"[CLASSES] first 8 model classes = {model_class_names[:8]}")

    return model_class_names


def build_submission(
    test_files,
    session,
    input_name,
    mel_converter,
    official_cols,
    target_cols,
    model_class_names,
):
    num_model_classes = len(model_class_names)

    if len(target_cols) != num_model_classes:
        raise ValueError(
            f"Number of official target columns ({len(target_cols)}) "
            f"!= number of model classes ({num_model_classes})."
        )

    missing_in_model = sorted(set(target_cols) - set(model_class_names))
    extra_in_model = sorted(set(model_class_names) - set(target_cols))

    if missing_in_model or extra_in_model:
        print(f"[ERROR] missing_in_model count = {len(missing_in_model)}")
        print(f"[ERROR] extra_in_model count = {len(extra_in_model)}")
        print(f"[ERROR] first missing = {missing_in_model[:10]}")
        print(f"[ERROR] first extra = {extra_in_model[:10]}")
        raise ValueError("Official target columns and model class names do not match.")

    if model_class_names == target_cols:
        print("[CLASSES] Model class order exactly matches sample_submission target order.")
    else:
        print("[CLASSES] Model class set matches official targets, but order differs.")
        print("[CLASSES] DataFrame column reindexing will align output to official order.")

    rows = []

    start_time = time.time()

    for file_idx, file_path in enumerate(test_files, start=1):
        stem = os.path.splitext(os.path.basename(file_path))[0]

        if file_idx <= 5 or file_idx % 50 == 0:
            print(f"[RUN] {file_idx}/{len(test_files)}: {stem}")

        probs = predict_one_file(
            file_path=file_path,
            session=session,
            input_name=input_name,
            mel_converter=mel_converter,
            num_model_classes=num_model_classes,
        )

        # 固定 12 rows: stem_5, stem_10, ..., stem_60
        for window_idx, end_time in enumerate(range(5, 65, 5)):
            row = {"row_id": f"{stem}_{end_time}"}

            # 先用 model_class_names 建欄位，最後再依 official_cols 重排
            for class_idx, class_name in enumerate(model_class_names):
                row[class_name] = float(probs[window_idx, class_idx])

            rows.append(row)

    submission_df = pd.DataFrame(rows)

    # 強制使用 sample_submission 欄位順序。
    # 這一步同時會把 target columns 重新排成官方順序。
    submission_df = submission_df[official_cols]

    # 強制數值欄位格式乾淨
    values = submission_df[target_cols].to_numpy(dtype=np.float32)
    values = np.nan_to_num(values, nan=0.0, posinf=1.0, neginf=0.0)
    values = np.clip(values, 0.0, 1.0)
    submission_df[target_cols] = values.astype(np.float32)

    elapsed = time.time() - start_time
    print(f"[RUN] Inference finished in {elapsed:.2f} seconds.")

    return submission_df


def validate_submission(submission_df, test_files, official_cols, target_cols):
    """
    Kaggle scoring error 常見原因：
        - row 數錯
        - 欄位數錯
        - 欄位順序錯
        - row_id 缺失 / 重複 / 多出 _65
        - NaN / inf
        - target columns 非數值
    """
    print("\n" + "=" * 80)
    print("Submission validation")
    print("=" * 80)

    expected_rows = len(test_files) * N_WINDOWS
    expected_cols = len(official_cols)

    print(f"[CHECK] submission shape = {submission_df.shape}")
    print(f"[CHECK] expected rows = {expected_rows}")
    print(f"[CHECK] expected cols = {expected_cols}")

    if len(test_files) > 0:
        expected_row_ids = [
            f"{os.path.splitext(os.path.basename(p))[0]}_{t}"
            for p in test_files
            for t in range(5, 65, 5)
        ]

        print(f"[CHECK] first 12 expected row_id = {expected_row_ids[:12]}")
        print(f"[CHECK] first 12 actual row_id   = {submission_df['row_id'].head(12).tolist()}")

        assert submission_df["row_id"].tolist() == expected_row_ids, (
            "row_id order/content mismatch. "
            "This usually means file sorting or window generation is wrong."
        )

    assert submission_df.shape[0] == expected_rows, (
        f"Wrong row count: got {submission_df.shape[0]}, expected {expected_rows}"
    )

    assert submission_df.shape[1] == expected_cols, (
        f"Wrong column count: got {submission_df.shape[1]}, expected {expected_cols}"
    )

    assert submission_df.columns.tolist() == official_cols, (
        "Column order mismatch against sample_submission.csv."
    )

    assert submission_df["row_id"].notna().all(), "row_id contains NaN."
    assert submission_df["row_id"].is_unique, "row_id contains duplicates."

    bad_65 = submission_df["row_id"].astype(str).str.endswith("_65").sum()
    assert bad_65 == 0, "Found row_id ending with _65. Each file must stop at _60."

    pred_values = submission_df[target_cols].to_numpy()
    assert np.isfinite(pred_values).all(), "Prediction matrix contains NaN or inf."

    min_value = float(pred_values.min()) if pred_values.size else None
    max_value = float(pred_values.max()) if pred_values.size else None

    print(f"[CHECK] prediction min = {min_value}")
    print(f"[CHECK] prediction max = {max_value}")
    print("[CHECK] validation passed.")
    print("=" * 80 + "\n")


def write_placeholder_submission_if_no_test_files(sample_df):
    """
    Edit mode 若 test_soundscapes 為空，仍輸出一份合法 CSV。
    注意：Submit mode 如果也抓不到 test files，通常代表 TEST_AUDIO_DIR 路徑或掛載有問題。
    """
    print("[WARN] No test files found.")
    print("[WARN] Writing sample_submission.csv as placeholder.")
    print("[WARN] If this happens during official submit, check TEST_AUDIO_DIR path immediately.")

    sample_df.to_csv(SUBMISSION_PATH, index=False)
    print(f"[OK] Placeholder submission saved to {SUBMISSION_PATH}")
    print(f"[OK] Placeholder shape = {sample_df.shape}")


# ============================================================
# 8. Main
# ============================================================

def main():
    total_start = time.time()

    configure_torch_threads()

    print(f"[INFO] Python executable = {sys.executable}")
    print(f"[INFO] os.cpu_count() = {os.cpu_count()}")
    print(f"[INFO] NUM_THREADS = {NUM_THREADS}")
    print(f"[INFO] torch num threads = {torch.get_num_threads()}")
    print(f"[INFO] SAMPLE_RATE = {SAMPLE_RATE}")
    print(f"[INFO] FILE_DURATION = {FILE_DURATION}")
    print(f"[INFO] N_WINDOWS per file = {N_WINDOWS}")
    print(f"[INFO] CHUNK_SAMPLES = {CHUNK_SAMPLES}")
    print(f"[INFO] BATCH_SIZE = {BATCH_SIZE}")

    test_files = validate_environment()

    ort = ensure_onnxruntime()

    sample_df, official_cols, target_cols = load_submission_schema()
    model_class_names = load_model_class_names()
    
    # Schema-level hard checks
    assert len(target_cols) == len(model_class_names), (
        f"target_cols={len(target_cols)}, model_class_names={len(model_class_names)}"
    )
    
    assert set(target_cols) == set(model_class_names), (
        "sample_submission target columns and taxonomy primary_label set do not match."
    )
    
    if target_cols == model_class_names:
        print("[SCHEMA] target column order exactly matches model_class_names.")
    else:
        print("[SCHEMA] target column order differs from model_class_names.")
        print("[SCHEMA] This is acceptable only if model_class_names reflects the actual model output order.")
    
    # 重要：即使沒有 test files，也先驗證 ONNX session 與一次 dummy inference。
    session, input_name, output_names = build_onnx_session(ort)
    mel_converter = AudioToMelSpectrogram().eval()
    
    run_preflight_inference(
        session=session,
        input_name=input_name,
        mel_converter=mel_converter,
        num_model_classes=len(model_class_names),
    )
    
    # 若 edit mode 沒有 test files，才輸出 sample_submission placeholder。
    if len(test_files) == 0:
        write_placeholder_submission_if_no_test_files(sample_df)
        return

    submission_df = build_submission(
        test_files=test_files,
        session=session,
        input_name=input_name,
        mel_converter=mel_converter,
        official_cols=official_cols,
        target_cols=target_cols,
        model_class_names=model_class_names,
    )

    validate_submission(
        submission_df=submission_df,
        test_files=test_files,
        official_cols=official_cols,
        target_cols=target_cols,
    )

    submission_df.to_csv(SUBMISSION_PATH, index=False)

    print(f"[OK] submission.csv saved to: {SUBMISSION_PATH}")
    print(f"[OK] final shape = {submission_df.shape}")
    print("[OK] preview:")
    print(submission_df.iloc[:5, :8])

    total_elapsed = time.time() - total_start
    print(f"[OK] total elapsed time = {total_elapsed:.2f} seconds")


if __name__ == "__main__":
    main()

[INFO] torch.set_num_threads(4) succeeded.
[INFO] torch.set_num_interop_threads(1) succeeded.
[INFO] Python executable = /usr/bin/python3
[INFO] os.cpu_count() = 4
[INFO] NUM_THREADS = 4
[INFO] torch num threads = 4
[INFO] SAMPLE_RATE = 32000
[INFO] FILE_DURATION = 60
[INFO] N_WINDOWS per file = 12
[INFO] CHUNK_SAMPLES = 160000
[INFO] BATCH_SIZE = 16

Environment / path validation
[PATH] WHEEL_DIR
       path    = /kaggle/input/datasets/kevinlin0411/kaggle-onnx-wheels/onnx_wheels
       exists  = True
       is_file = False
       is_dir  = True
[PATH] TEST_AUDIO_DIR
       path    = /kaggle/input/competitions/birdclef-2026/test_soundscapes
       exists  = True
       is_file = False
       is_dir  = True
[PATH] WEIGHTS_DIR
       path    = /kaggle/input/datasets/kevinlin0411/birdclef-2026-my-models
       exists  = True
       is_file = False
       is_dir  = True
[PATH] TAXA_CSV
       path    = /kaggle/input/competitions/birdclef-2026/taxonomy.csv
       exists  = True
       is_fi